In [4]:
"""
College Indoor Navigation System
Pure Python, terminal-based, graph + A* pathfinding
"""

import heapq
import json
from dataclasses import dataclass, field
from typing import Optional


# ─────────────────────────────────────────────────────────────
# DATA CLASSES
# ─────────────────────────────────────────────────────────────

@dataclass
class Location:
    id: int
    name: str
    floor: str
    type: str          # Lab, Room, Office, Washroom, Staircase, Elevator, etc.

    def to_dict(self):
        return {"id": self.id, "name": self.name, "floor": self.floor, "type": self.type}


@dataclass(order=True)
class _PQItem:
    """Priority-queue item for A* / Dijkstra."""
    priority: float
    node_id: int = field(compare=False)


# ─────────────────────────────────────────────────────────────
# GRAPH
# ─────────────────────────────────────────────────────────────

class Graph:
    """Weighted directed graph with navigation instructions on edges."""

    def __init__(self):
        self.locations: dict[int, Location] = {}          # id → Location
        self._adj: dict[int, list[tuple[int, float, str]]] = {}  # id → [(neighbour_id, cost, instruction)]

    # ── Construction helpers ──────────────────────────────────

    def add_location(self, location: Location):
        self.locations[location.id] = location
        if location.id not in self._adj:
            self._adj[location.id] = []

    def add_edge(self, from_id: int, to_id: int, cost: float, instruction: str, bidirectional: bool = True):
        self._adj[from_id].append((to_id, cost, instruction))
        if bidirectional:
            # Reverse instruction for the back-direction
            rev = self._reverse_instruction(instruction)
            self._adj[to_id].append((from_id, cost, rev))

    # ── Routing ──────────────────────────────────────────────

    def dijkstra(self, start: int, end: int) -> tuple[list[int], list[str]]:
        """Returns (node_id_path, instruction_list) or ([], []) if unreachable."""
        dist = {nid: float("inf") for nid in self.locations}
        dist[start] = 0
        prev: dict[int, Optional[int]] = {nid: None for nid in self.locations}
        prev_instr: dict[int, str] = {}

        pq = [_PQItem(0, start)]
        visited = set()

        while pq:
            item = heapq.heappop(pq)
            cur = item.node_id
            if cur in visited:
                continue
            visited.add(cur)
            if cur == end:
                break
            for nbr, cost, instr in self._adj.get(cur, []):
                new_dist = dist[cur] + cost
                if new_dist < dist[nbr]:
                    dist[nbr] = new_dist
                    prev[nbr] = cur
                    prev_instr[nbr] = instr
                    heapq.heappush(pq, _PQItem(new_dist, nbr))

        if dist[end] == float("inf"):
            return [], []

        # Reconstruct path
        path, instrs = [], []
        cur = end
        while cur is not None:
            path.append(cur)
            if cur in prev_instr:
                instrs.append(prev_instr[cur])
            cur = prev[cur]
        path.reverse()
        instrs.reverse()
        return path, instrs

    # ── Heuristic (floor distance) ─────────────────────────

    _FLOOR_ORDER = {"Cellar": 0, "Ground": 1, "First": 2, "Second": 3, "Third": 4, "Fourth": 5}

    def _heuristic(self, a: int, b: int) -> float:
        fa = self._FLOOR_ORDER.get(self.locations[a].floor, 0)
        fb = self._FLOOR_ORDER.get(self.locations[b].floor, 0)
        return abs(fa - fb) * 0.5   # small admissible heuristic

    def astar(self, start: int, end: int) -> tuple[list[int], list[str]]:
        """A* search; falls back gracefully to Dijkstra if heuristic is zero."""
        g_score = {nid: float("inf") for nid in self.locations}
        g_score[start] = 0
        f_score = {nid: float("inf") for nid in self.locations}
        f_score[start] = self._heuristic(start, end)

        prev: dict[int, Optional[int]] = {nid: None for nid in self.locations}
        prev_instr: dict[int, str] = {}

        pq = [_PQItem(f_score[start], start)]
        visited = set()

        while pq:
            item = heapq.heappop(pq)
            cur = item.node_id
            if cur in visited:
                continue
            visited.add(cur)
            if cur == end:
                break
            for nbr, cost, instr in self._adj.get(cur, []):
                tentative = g_score[cur] + cost
                if tentative < g_score[nbr]:
                    g_score[nbr] = tentative
                    f_score[nbr] = tentative + self._heuristic(nbr, end)
                    prev[nbr] = cur
                    prev_instr[nbr] = instr
                    heapq.heappush(pq, _PQItem(f_score[nbr], nbr))

        if g_score[end] == float("inf"):
            return [], []

        path, instrs = [], []
        cur = end
        while cur is not None:
            path.append(cur)
            if cur in prev_instr:
                instrs.append(prev_instr[cur])
            cur = prev[cur]
        path.reverse()
        instrs.reverse()
        return path, instrs

    # ── Utilities ─────────────────────────────────────────────

    @staticmethod
    def _reverse_instruction(instr: str) -> str:
        """Best-effort reversal of a direction string."""
        rev_map = {
            "Turn left": "Turn right",
            "Turn right": "Turn left",
            "turn left": "turn right",
            "turn right": "turn left",
        }
        for k, v in rev_map.items():
            if k in instr:
                return instr.replace(k, v)
        return instr   # symmetric instructions (straight, continue) remain unchanged

    def to_dict(self) -> dict:
        """Serialise graph to a JSON-friendly dict for future JSON loading."""
        return {
            "locations": [loc.to_dict() for loc in self.locations.values()],
            "edges": [
                {"from": fid, "to": tid, "cost": cost, "instruction": instr}
                for fid, neighbours in self._adj.items()
                for tid, cost, instr in neighbours
            ],
        }

    @classmethod
    def from_dict(cls, data: dict) -> "Graph":
        g = cls()
        for loc in data["locations"]:
            g.add_location(Location(**loc))
        for edge in data["edges"]:
            # When loading from JSON we add directed edges directly (already expanded)
            g._adj[edge["from"]].append((edge["to"], edge["cost"], edge["instruction"]))
        return g


# ─────────────────────────────────────────────────────────────
# NAVIGATION SYSTEM
# ─────────────────────────────────────────────────────────────

class NavigationSystem:
    def __init__(self, graph: Graph):
        self.graph = graph

    # ── Route finding ─────────────────────────────────────────

    def find_route(self, source_id: int, dest_id: int) -> tuple[list[int], list[str]]:
        return self.graph.astar(source_id, dest_id)

    # ── Direction generation ───────────────────────────────────

    def generate_directions(self, path: list[int], instructions: list[str]) -> list[str]:
        """Turn raw instructions + node path into numbered human-readable steps."""
        locs = self.graph.locations
        directions = []

        step = 1
        for i, instr in enumerate(instructions):
            directions.append(f"{step}. {instr}")
            step += 1

        if path:
            dest = locs[path[-1]]
            directions.append(f"{step}. You have arrived at your destination: {dest.name}.")

        return directions

    # ── Display helpers ────────────────────────────────────────

    def display_locations(self, exclude_id: Optional[int] = None) -> dict[int, int]:
        """
        Print numbered location list, return mapping display_number → location_id.
        Optionally exclude one location (the already-chosen source).
        """
        locs = [loc for loc in sorted(self.graph.locations.values(), key=lambda l: l.id)
                if loc.id != exclude_id]

        mapping = {}
        for display_num, loc in enumerate(locs, start=1):
            floor_label = f"[{loc.floor} Floor]" if loc.floor not in ("Cellar",) else "[Cellar]"
            print(f"  [{display_num:>3}] {loc.name:<40} {floor_label}")
            mapping[display_num] = loc.id

        return mapping

    def get_user_choice(self, mapping: dict[int, int], prompt: str) -> int:
        """Prompt until the user enters a valid display number; return location_id."""
        while True:
            try:
                choice = int(input(f"\n{prompt}: ").strip())
                if choice in mapping:
                    return mapping[choice]
                print(f"  ✗  Please enter a number between 1 and {max(mapping)}.")
            except ValueError:
                print("  ✗  Invalid input. Enter a number.")

    # ── Main loop ─────────────────────────────────────────────

    def run(self):
        self._print_banner()

        # ── Step 1: Source ────────────────────────────────────
        print("\n  Select Source Location\n")
        src_map = self.display_locations()
        source_id = self.get_user_choice(src_map, "Enter Source Number")
        source = self.graph.locations[source_id]

        # ── Step 2: Destination ───────────────────────────────
        print(f"\n  Source Selected: {source.name}  [{source.floor}]")
        print("\n  Select Destination Location\n")
        dst_map = self.display_locations(exclude_id=source_id)
        dest_id = self.get_user_choice(dst_map, "Enter Destination Number")
        dest = self.graph.locations[dest_id]

        # ── Step 3: Pathfinding ───────────────────────────────
        print("\n  Calculating route …")
        path, instructions = self.find_route(source_id, dest_id)

        # ── Step 4: Output ────────────────────────────────────
        print("\n" + "=" * 50)
        if not path:
            print("  ✗  No route found between the selected locations.")
        else:
            print("  ROUTE FOUND")
            print("=" * 50)
            print(f"\n  Source      : {source.name}  [{source.floor} Floor]")
            print(f"  Destination : {dest.name}  [{dest.floor} Floor]")
            print("\n  Directions:\n")
            directions = self.generate_directions(path, instructions)
            for d in directions:
                print(f"    {d}")
            print("\n  ✔  Destination Arrived.\n")
        print("=" * 50)

    @staticmethod
    def _print_banner():
        print()
        print("=" * 50)
        print("     COLLEGE INDOOR NAVIGATION SYSTEM")
        print("=" * 50)


# ─────────────────────────────────────────────────────────────
# BUILDING DATA – Graph construction
# ─────────────────────────────────────────────────────────────

def build_college_graph() -> Graph:
    g = Graph()

    # ── Helper shortcuts ──────────────────────────────────────
    def loc(id_, name, floor, type_):
        g.add_location(Location(id_, name, floor, type_))

    def edge(a, b, cost, instr):
        g.add_edge(a, b, cost, instr)

    # ─────────────────────────────────────────────────────────
    # CELLAR FLOOR  (IDs 100-149)
    # ─────────────────────────────────────────────────────────
    loc(100, "Cellar Entrance / Elevator", "Cellar", "Elevator")
    loc(101, "Girls Common Room",          "Cellar", "CommonRoom")
    loc(102, "Boys Common Room",           "Cellar", "CommonRoom")
    loc(103, "Examination Branch",         "Cellar", "Office")
    loc(104, "Lift 2 (Cellar)",            "Cellar", "Elevator")
    loc(105, "Staircase Set 2 (Cellar)",   "Cellar", "Staircase")
    loc(106, "HC01 IoT Lab",               "Cellar", "Lab")
    loc(107, "Workshop 1",                 "Cellar", "Lab")
    loc(108, "Workshop 2",                 "Cellar", "Lab")
    loc(109, "Workshop 3",                 "Cellar", "Lab")
    loc(110, "Room 14C",                   "Cellar", "Room")
    loc(111, "Room 14B",                   "Cellar", "Room")
    loc(112, "Room 14A",                   "Cellar", "Room")
    loc(113, "Room 15A",                   "Cellar", "Room")
    loc(114, "Room 15B",                   "Cellar", "Room")
    loc(115, "Room 15C",                   "Cellar", "Room")
    loc(116, "Room 15D",                   "Cellar", "Room")
    loc(117, "Staircase to Ground Floor",  "Cellar", "Staircase")

    # Left-side corridor: Elevator → GCR → BCR → Exam → Lift2 → Staircase2 → IoT Lab
    edge(100, 101, 1, "Walk straight along the left corridor past the elevator to reach Girls Common Room.")
    edge(101, 102, 1, "Continue straight along the corridor to reach Boys Common Room.")
    edge(102, 103, 1, "Continue forward along the left corridor to reach the Examination Branch.")
    edge(103, 104, 1, "Continue straight to reach Lift 2.")
    edge(104, 105, 1, "Continue forward to reach Staircase Set 2.")
    edge(105, 106, 1, "Continue straight to reach HC01 IoT Lab.")

    # Right-side corridor: Workshops and classrooms
    edge(100, 107, 1, "Turn right from the elevator and walk along the right corridor to Workshop 1.")
    edge(107, 108, 1, "Continue forward on the right side to Workshop 2.")
    edge(108, 109, 1, "Continue forward to Workshop 3.")
    edge(109, 110, 1, "Continue forward to Room 14C.")
    edge(110, 111, 1, "Continue along the right corridor to Room 14B.")
    edge(111, 112, 1, "Continue forward to Room 14A.")
    edge(112, 113, 1, "Continue forward to Room 15A.")
    edge(113, 114, 1, "Continue forward to Room 15B.")
    edge(114, 115, 1, "Continue forward to Room 15C.")
    edge(115, 116, 1, "Continue forward to Room 15D.")
    edge(116, 117, 1, "Continue to the end of the corridor to reach the Staircase to Ground Floor.")

    # Cross-links (left ↔ right at key points)
    edge(106, 117, 2, "Walk across to the Staircase to Ground Floor at the end of the cellar.")
    edge(105, 112, 2, "Cut across the cellar corridor to reach Room 14A.")

    # ─────────────────────────────────────────────────────────
    # GROUND FLOOR  (IDs 200-229)
    # ─────────────────────────────────────────────────────────
    loc(200, "Ground Floor Landing",       "Ground", "Corridor")
    loc(201, "Principal Office",           "Ground", "Office")
    loc(202, "Board Room",                 "Ground", "Office")
    loc(203, "Staircase to First Floor",   "Ground", "Staircase")
    loc(204, "Girls Washroom (Ground)",    "Ground", "Washroom")
    loc(205, "Room 003",                   "Ground", "Room")
    loc(206, "Room 004",                   "Ground", "Room")
    loc(207, "Room 005",                   "Ground", "Room")
    loc(208, "Room 006",                   "Ground", "Room")
    loc(209, "Lab 008",                    "Ground", "Lab")
    loc(210, "Boys Washroom (Ground)",     "Ground", "Washroom")
    loc(211, "Elevator (Ground)",          "Ground", "Elevator")
    loc(212, "Staircase Set 1 (Ground)",   "Ground", "Staircase")
    loc(213, "Room 007",                   "Ground", "Room")

    # Staircase from Cellar arrives at Ground Landing
    edge(117, 200, 2, "Climb the staircase from the Cellar to reach the Ground Floor landing.")

    # Ground floor corridor
    edge(200, 201, 1, "From the landing, walk right toward the Principal Office.")
    edge(201, 202, 1, "Continue forward to the Board Room.")
    edge(202, 203, 1, "Continue to the Staircase to the First Floor.")
    edge(203, 204, 1, "Walk further along the corridor past the staircase to the Girls Washroom.")
    edge(204, 205, 1, "Continue forward to Room 003.")
    edge(205, 206, 1, "Continue forward to Room 004.")
    edge(206, 207, 1, "Continue forward to Room 005.")
    edge(207, 208, 1, "Continue forward to Room 006.")
    edge(208, 209, 1, "Near Room 006, turn to find Lab 008.")
    edge(208, 210, 1, "Near Room 006, the Boys Washroom is just ahead.")
    edge(210, 211, 1, "Continue to the Elevator on the Ground Floor.")
    edge(211, 212, 1, "Continue forward to Staircase Set 1.")
    edge(212, 213, 1, "Continue to Room 007 further along the corridor.")

    # ─────────────────────────────────────────────────────────
    # FIRST FLOOR  (IDs 300-329)
    # ─────────────────────────────────────────────────────────
    loc(300, "First Floor Landing",        "First", "Corridor")
    loc(301, "Room 107",                   "First", "Room")
    loc(302, "Lab 108",                    "First", "Lab")
    loc(303, "Elevator (First)",           "First", "Elevator")
    loc(304, "Boys Washroom (First)",      "First", "Washroom")
    loc(305, "Room 106",                   "First", "Room")
    loc(306, "Room 105",                   "First", "Room")
    loc(307, "Room 104",                   "First", "Room")
    loc(308, "Room 103",                   "First", "Room")
    loc(309, "Girls Washroom (First)",     "First", "Washroom")
    loc(310, "Lift 2 (First)",             "First", "Elevator")
    loc(311, "Staircase Set 2 (First)",    "First", "Staircase")
    loc(312, "Room 102",                   "First", "Room")
    loc(313, "Room 101",                   "First", "Room")
    loc(314, "Staff Room (First)",         "First", "Office")
    loc(315, "FED HOD Room",               "First", "Office")
    loc(316, "Staircase Set 3 (First)",    "First", "Staircase")
    loc(317, "Staircase Set 1 (First)",    "First", "Staircase")

    # Connect Ground Set1 → First floor landing
    edge(212, 317, 2, "Take Staircase Set 1 up to the First Floor.")
    edge(203, 300, 2, "Climb the staircase to the First Floor landing.")

    edge(300, 301, 1, "From the First Floor landing, turn right to reach Room 107.")
    edge(300, 302, 1, "Walk straight ahead from the landing to Lab 108.")
    edge(300, 303, 1, "Turn left from the landing to reach the Elevator.")
    edge(303, 304, 1, "Continue left past the elevator to the Boys Washroom.")
    edge(304, 305, 1, "Continue along the left corridor to Room 106.")
    edge(305, 306, 1, "Continue forward to Room 105.")
    edge(306, 307, 1, "Continue forward to Room 104.")
    edge(307, 308, 1, "Continue forward to Room 103.")
    edge(308, 309, 1, "Continue to the Girls Washroom.")
    edge(309, 310, 1, "Continue forward to Lift 2.")
    edge(310, 311, 1, "Continue to Staircase Set 2.")
    edge(311, 312, 1, "Continue forward to Room 102.")
    edge(312, 313, 1, "Continue to Room 101.")
    edge(313, 314, 1, "Turn right to find the Staff Room.")
    edge(314, 315, 1, "Continue to the FED HOD Room at the end.")
    edge(315, 316, 1, "Staircase Set 3 is at the end of the corridor near FED HOD.")
    edge(317, 300, 1, "Walk from Staircase Set 1 onto the First Floor landing.")

    # ─────────────────────────────────────────────────────────
    # SECOND FLOOR  (IDs 400-429)
    # ─────────────────────────────────────────────────────────
    loc(400, "Second Floor Landing",       "Second", "Corridor")
    loc(401, "ECE HOD Room",               "Second", "Office")
    loc(402, "ECE Staff Room",             "Second", "Office")
    loc(403, "Room 201",                   "Second", "Room")
    loc(404, "Room 202",                   "Second", "Room")
    loc(405, "Lift 2 (Second)",            "Second", "Elevator")
    loc(406, "Staircase Set 2 (Second)",   "Second", "Staircase")
    loc(407, "Room 203",                   "Second", "Room")
    loc(408, "Room 204",                   "Second", "Room")
    loc(409, "Room 205",                   "Second", "Room")
    loc(410, "Room 206",                   "Second", "Room")
    loc(411, "Boys Washroom (Second)",     "Second", "Washroom")
    loc(412, "Elevator (Second)",          "Second", "Elevator")
    loc(413, "Staircase Set 1 (Second)",   "Second", "Staircase")
    loc(414, "Library",                    "Second", "Library")

    edge(317, 413, 2, "Continue up Staircase Set 1 to the Second Floor.")
    edge(316, 406, 2, "Take Staircase Set 3 / Set 2 up to the Second Floor.")

    edge(400, 401, 1, "Turn right from the landing to the ECE HOD Room.")
    edge(401, 402, 1, "Continue right to the ECE Staff Room.")
    edge(400, 403, 1, "Turn left from the landing to Room 201.")
    edge(403, 404, 1, "Continue forward to Room 202.")
    edge(404, 405, 1, "Continue to Lift 2.")
    edge(405, 406, 1, "Continue to Staircase Set 2.")
    edge(406, 407, 1, "Continue forward to Room 203.")
    edge(407, 408, 1, "Continue to Room 204.")
    edge(408, 409, 1, "Continue to Room 205.")
    edge(409, 410, 1, "Continue to Room 206.")
    edge(410, 411, 1, "Turn right to the Boys Washroom.")
    edge(411, 412, 1, "Continue forward to the Elevator.")
    edge(412, 413, 1, "Continue to Staircase Set 1.")
    edge(413, 414, 1, "Continue forward past Staircase Set 1 to reach the Library.")
    edge(413, 400, 1, "Walk from Staircase Set 1 onto the Second Floor main corridor.")

    # ─────────────────────────────────────────────────────────
    # THIRD FLOOR  (IDs 500-529)
    # ─────────────────────────────────────────────────────────
    loc(500, "Third Floor Landing",        "Third", "Corridor")
    loc(501, "Room 307",                   "Third", "Room")
    loc(502, "Auditorium Entrance",        "Third", "Landmark")
    loc(503, "Elevator (Third)",           "Third", "Elevator")
    loc(504, "Boys Washroom (Third)",      "Third", "Washroom")
    loc(505, "Room 306",                   "Third", "Room")
    loc(506, "Room 305",                   "Third", "Room")
    loc(507, "Room 304",                   "Third", "Room")
    loc(508, "Lab H303",                   "Third", "Lab")
    loc(509, "Girls Washroom (Third)",     "Third", "Washroom")
    loc(510, "Lift 2 (Third)",             "Third", "Elevator")
    loc(511, "Staircase Set 2 (Third)",    "Third", "Staircase")
    loc(512, "Room 302",                   "Third", "Room")
    loc(513, "Lab H301",                   "Third", "Lab")
    loc(514, "Staff Room (Third)",         "Third", "Office")
    loc(515, "CSIT HOD Room",              "Third", "Office")
    loc(516, "Staircase Set 3 (Third)",    "Third", "Staircase")
    loc(517, "Staircase Set 1 (Third)",    "Third", "Staircase")

    edge(413, 517, 2, "Climb Staircase Set 1 to the Third Floor.")
    edge(406, 511, 2, "Climb Staircase Set 2 to the Third Floor.")
    edge(316, 516, 4, "Take Staircase Set 3 up to the Third Floor.")

    edge(500, 501, 1, "Turn right from the landing to Room 307.")
    edge(500, 502, 1, "Walk straight ahead — Auditorium Entrance is directly in front of you.")
    edge(500, 503, 1, "Turn left from the landing to the Elevator.")
    edge(503, 504, 1, "Continue left past the elevator to the Boys Washroom.")
    edge(504, 505, 1, "Continue along the left corridor to Room 306.")
    edge(505, 506, 1, "Continue forward to Room 305.")
    edge(506, 507, 1, "Continue forward to Room 304.")
    edge(507, 508, 1, "Continue forward to Lab H303.")
    edge(508, 509, 1, "Continue to the Girls Washroom.")
    edge(509, 510, 1, "Continue forward to Lift 2.")
    edge(510, 511, 1, "Continue to Staircase Set 2.")
    edge(511, 512, 1, "Continue forward to Room 302.")
    edge(512, 513, 1, "Continue to Lab H301.")
    edge(513, 514, 1, "Turn right to the Staff Room.")
    edge(514, 515, 1, "Continue to the CSIT HOD Room.")
    edge(515, 516, 1, "Staircase Set 3 is at the end of the corridor.")
    edge(517, 500, 1, "Walk from Staircase Set 1 onto the Third Floor landing.")

    # ─────────────────────────────────────────────────────────
    # FOURTH FLOOR  (IDs 600-629)
    # ─────────────────────────────────────────────────────────
    loc(600, "Fourth Floor Landing",       "Fourth", "Corridor")
    loc(601, "CSE HOD Room",               "Fourth", "Office")
    loc(602, "CSE Staff Room",             "Fourth", "Office")
    loc(603, "Room 401",                   "Fourth", "Room")
    loc(604, "Room 402",                   "Fourth", "Room")
    loc(605, "Staircase Set 2 (Fourth)",   "Fourth", "Staircase")
    loc(606, "Girls Washroom (Fourth)",    "Fourth", "Washroom")
    loc(607, "Room 403",                   "Fourth", "Room")
    loc(608, "Room 404",                   "Fourth", "Room")
    loc(609, "Room 405",                   "Fourth", "Room")
    loc(610, "Room 406",                   "Fourth", "Room")
    loc(611, "Boys Washroom (Fourth)",     "Fourth", "Washroom")
    loc(612, "Elevator (Fourth)",          "Fourth", "Elevator")
    loc(613, "Staircase Set 1 (Fourth)",   "Fourth", "Staircase")
    loc(614, "Room 407",                   "Fourth", "Room")
    loc(615, "Auditorium Exit",            "Fourth", "Landmark")

    edge(517, 613, 2, "Climb Staircase Set 1 from Third to Fourth Floor.")
    edge(511, 605, 2, "Climb Staircase Set 2 from Third to Fourth Floor.")

    edge(600, 601, 1, "Turn right from the landing to the CSE HOD Room.")
    edge(601, 602, 1, "Continue to the CSE Staff Room.")
    edge(600, 603, 1, "Turn left from the landing to Room 401.")
    edge(603, 604, 1, "Continue to Room 402.")
    edge(604, 605, 1, "Continue forward to Staircase Set 2.")
    edge(605, 606, 1, "Continue to the Girls Washroom.")
    edge(606, 607, 1, "Continue forward to Room 403.")
    edge(607, 608, 1, "Continue to Room 404.")
    edge(608, 609, 1, "Continue to Room 405.")
    edge(609, 610, 1, "Continue to Room 406.")
    edge(610, 611, 1, "Continue to the Boys Washroom.")
    edge(611, 612, 1, "Continue to the Elevator.")
    edge(612, 613, 1, "Continue to Staircase Set 1.")
    edge(613, 614, 1, "Continue past the staircase to Room 407.")
    edge(614, 615, 1, "Continue to the Auditorium Exit at the end of the corridor.")
    edge(613, 600, 1, "Walk from Staircase Set 1 onto the Fourth Floor landing.")

    # ─────────────────────────────────────────────────────────
    # VERTICAL CONNECTIONS – Elevators & Lifts  (shared shafts)
    # ─────────────────────────────────────────────────────────
    elevator_ids  = [100, 211, 303, 412, 503, 612]
    lift2_ids     = [104, 310, 405, 510]

    for shaft, cost, verb in [(elevator_ids, 3, "Take the elevator"), (lift2_ids, 3, "Take Lift 2")]:
        for i in range(len(shaft) - 1):
            a, b = shaft[i], shaft[i + 1]
            la   = g.locations[a]
            lb   = g.locations[b]
            g.add_edge(a, b, cost,
                       f"{verb} from the {la.floor} Floor up to the {lb.floor} Floor.")

    return g


# ─────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────

def main():
    graph = build_college_graph()
    nav   = NavigationSystem(graph)

    while True:
        nav.run()
        print()
        again = input("  Navigate again? (y / n): ").strip().lower()
        if again != "y":
            print("\n  Thank you for using the College Navigation System. Goodbye!\n")
            break


if __name__ == "__main__":
    main()


     COLLEGE INDOOR NAVIGATION SYSTEM

  Select Source Location

  [  1] Cellar Entrance / Elevator               [Cellar]
  [  2] Girls Common Room                        [Cellar]
  [  3] Boys Common Room                         [Cellar]
  [  4] Examination Branch                       [Cellar]
  [  5] Lift 2 (Cellar)                          [Cellar]
  [  6] Staircase Set 2 (Cellar)                 [Cellar]
  [  7] HC01 IoT Lab                             [Cellar]
  [  8] Workshop 1                               [Cellar]
  [  9] Workshop 2                               [Cellar]
  [ 10] Workshop 3                               [Cellar]
  [ 11] Room 14C                                 [Cellar]
  [ 12] Room 14B                                 [Cellar]
  [ 13] Room 14A                                 [Cellar]
  [ 14] Room 15A                                 [Cellar]
  [ 15] Room 15B                                 [Cellar]
  [ 16] Room 15C                                 [Cellar]
  [ 17